Aim of this script: Clean Chuuchuu dataset before analysis.
Especially: 
- Identify correct countries (using db ID)
- Identify stations

In [1]:
import pandas as pd
import numpy as np

In [1]:
import_data = input("Should we import CSVs ? (Y/N)")

if import_data =="Y":
    data_stations = pd.read_csv("stations.csv",sep=";")#trainlain stations
    data_osm_stations = pd.read_csv("EU_train_stations_OSM.csv")#trainlain stations
    data_chuuchuu = pd.read_csv("delay_records_2026-02-26_to_2026-03-04.csv")

In [2]:
stations = data_stations
osm_stations = data_osm_stations 
chuuchuu = data_chuuchuu

NameError: name 'data_stations' is not defined

In [4]:
stations.head()

,id,name,slug,uic,uic8_sncf,latitude,longitude,parent_station_id,hub_id,country,...,info:ja,info:ko,info:pl,info:pt,info:ru,info:sv,info:tr,info:zh,normalised_code,iata_airport_code
0,1,Château-Arnoux – St-Auban,chateau-arnoux-st-auban,NaN,NaN,44.081790,6.001625,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv1,NaN
1,2,Château-Arnoux – St-Auban,chateau-arnoux-st-auban,8775123.0,87751230.0,44.061565,5.997373,1.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv2,NaN
2,3,Château-Arnoux Mairie,chateau-arnoux-mairie,8775122.0,87751222.0,44.063863,6.011248,1.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv3,NaN
3,4,Digne-les-Bains,digne-les-bains,NaN,NaN,44.350000,6.350000,NaN,NaN,FR,...,ディーニュ＝レ＝バン,디뉴레뱅,NaN,NaN,Динь-ле-Бен,NaN,NaN,迪涅萊班,urn:trainline:public:nloc:csv4,NaN
4,6,Digne-les-Bains,digne-les-bains,8775149.0,87751495.0,44.088710,6.222982,4.0,NaN,FR,...,ディーニュ＝レ＝バン,디뉴레뱅,NaN,NaN,Динь-ле-Бен,NaN,NaN,迪涅萊班,urn:trainline:public:nloc:csv6,NaN


In [5]:
chuuchuu.head()

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,arrivalPlatform,plannedArrivalPlatform,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra
0,NS,Sprinter,8148,2026-02-26,8400058,2026-02-25 04:51:47+00,Sprinter 8148,2992170,Amsterdam Centraal,2026-02-26 14:25:00+00,...,10a,10a,f,NaN,2026-02-26 14:25:00+00,NaN,10a,10a,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu..."
1,NS,Sprinter,8148,2026-02-26,8400059,2026-02-25 04:51:47+00,Sprinter 8148,2992236,Amsterdam Sloterdijk,2026-02-26 14:19:00+00,...,12,12,f,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00,0.0,12,12,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu..."
2,NS,Sprinter,8148,2026-02-26,8400561,2026-02-25 04:51:47+00,Sprinter 8148,3065389,Schiphol Airport,2026-02-26 14:07:00+00,...,3,3,f,2026-02-26 14:09:00+00,2026-02-26 14:09:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu..."
3,NS,Sprinter,8148,2026-02-26,8400079,2026-02-25 04:51:47+00,Sprinter 8148,2860888,Amsterdam Lelylaan,2026-02-26 14:15:00+00,...,2,2,f,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00,0.0,2,2,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu..."
4,NS,Sprinter,8148,2026-02-26,8400332,2026-02-25 04:51:47+00,Sprinter 8148,2993022,Hoofddorp,NaN,...,3,3,f,2026-02-26 14:02:00+00,2026-02-26 14:02:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu..."


In [6]:
stations["slug"]

0              chateau-arnoux-st-auban
1              chateau-arnoux-st-auban
2                chateau-arnoux-mairie
3                      digne-les-bains
4                      digne-les-bains
                     ...              
70841    la-defense-grande-arche-rer-e
70842     paris-gare-de-lyon-noctilien
70843        les-mureaux-vigne-blanche
70844                  basel-st-johann
70845                  basel-st-johann
Name: slug, Length: 70846, dtype: object

In [7]:
print(len(stations["slug"]))

70846


In [8]:
len(stations["slug"].unique())

69964

In [9]:
stations_unique = stations.drop_duplicates(subset="slug", keep="first")

In [12]:
len(stations_unique["slug"].unique())

69964

In [13]:
chuuchuu["stopName_slug"] = (
    chuuchuu["stopName"]
    .str.normalize("NFKD")                 # remove accents
    .str.encode("ascii", "ignore")
    .str.decode("utf-8")
    .str.lower()                            # lowercase
    .str.strip()                            # remove leading/trailing spaces
    # Replace "(" and "." with "-" and remove ")"
    .str.replace(r"\(", "-", regex=True)
    .str.replace(r"\.", "-", regex=True)
    .str.replace(r"\)", "", regex=True)
    # Replace any whitespace with "-"
    .str.replace(r"\s+", "-", regex=True)
    # Collapse multiple consecutive "-" into a single "-"
    .str.replace(r"-{2,}", "-", regex=True)
    # Remove leading/trailing "-" if any
    .str.strip("-")
)

In [14]:
chuuchuu["stopName_slug"]

0            amsterdam-centraal
1          amsterdam-sloterdijk
2              schiphol-airport
3            amsterdam-lelylaan
4                     hoofddorp
                   ...         
6623158                uzhhorod
6623159                  zahony
6623160             nyiregyhaza
6623161                 szolnok
6623162                debrecen
Name: stopName_slug, Length: 6623163, dtype: object

In [15]:
chuuchuu_merge = chuuchuu.merge(
    stations_unique[["slug", "country"]],
    how="left",                       # keep all rows in chuuchuu
    left_on="stopName_slug",            # column in chuuchuu
    right_on="slug"         # column in uic_codes
)
chuuchuu_merge = chuuchuu_merge.drop(columns=["slug"])

In [16]:
print(len(chuuchuu))
print(len(chuuchuu_merge))

6623163
6623163


In [17]:
# print(len(chuuchuu["countryStop"].unique()))
# print(chuuchuu["countryStop"].unique())
print(len(chuuchuu_merge["country"].unique()))
print(chuuchuu_merge["country"].unique())

22
['NL' 'IT' nan 'BE' 'DE' 'DK' 'FR' 'CH' 'HU' 'AT' 'PL' 'HR' 'CZ' 'RS' 'SI'
 'UA' 'LU' 'ES' 'RO' 'SK' 'SE' 'LI']


In [18]:
country_counts = (
    chuuchuu_merge.groupby("country")
    .agg(parent_count=("agency", lambda x: x.notna().sum()))
    .reset_index()
)
display(country_counts)
print(country_counts["parent_count"].sum())

,country,parent_count
0,AT,37000
1,BE,184139
2,CH,418970
3,CZ,2749
4,DE,2698898
5,DK,191645
6,ES,260
7,FR,374722
8,HR,567
9,HU,85560


4981797


In [19]:
chuuchuu_merge[chuuchuu_merge["country"].isna()]

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,stopName_slug,country
46,NS,Intercity,3142,2026-02-26,8400319,2026-02-25 13:23:47+00,Intercity 3142,3670859,'s-Hertogenbosch,2026-02-27 11:54:00+00,...,f,2026-02-27 11:55:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN
81,NS,Sprinter,8637,2026-02-26,8400053,2026-02-25 13:31:47+00,Sprinter 8637,2992146,Alphen a/d Rijn,NaN,...,f,2026-02-27 12:10:00+00,2026-02-26 12:10:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:R-NET_NS"", ""unplanned"": fals...",alphen-a/d-rijn,NaN
127,NS,Intercity,3171,2026-02-26,8400319,2026-02-25 13:31:47+00,Intercity 3171,3670859,'s-Hertogenbosch,2026-02-27 19:06:00+00,...,f,2026-02-27 19:08:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN
136,NS,Stoptrein,2583,2026-02-26,8800062,2026-02-25 13:31:47+00,Stoptrein 2583,2993451,Heide (B),2026-02-27 11:45:00+00,...,f,2026-02-27 11:46:00+00,2026-02-26 11:46:00+00,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NMBS"", ""unplanned"": false, ""...",heide-b,NaN
145,NS,Intercity,3143,2026-02-26,8400319,2026-02-25 13:31:47+00,Intercity 3143,3670859,'s-Hertogenbosch,2026-02-27 12:06:00+00,...,f,2026-02-27 12:08:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6623124,IT,IC,1963,2026-03-04,8300342,2026-03-05 16:25:51.205+00,IC 1963,S11774,VILLA S.GIOVANNI,2026-03-05 10:33:00+00,...,NaN,2026-03-05 10:40:00+00,2026-03-05 10:20:00+00,1200.0,NaN,NaN,NaN,"{""viaggiatrenoTripId"": ""S01700|1963|1772578800...",villa-s-giovanni,NaN
6623143,IT,IC,1963,2026-03-04,8300923,2026-03-05 16:25:51.205+00,IC 1963,S11727,SCALEA S.DOMENICA TALAO,2026-03-05 07:57:00+00,...,NaN,2026-03-05 08:00:00+00,2026-03-05 07:11:00+00,2940.0,NaN,3,NaN,"{""viaggiatrenoTripId"": ""S01700|1963|1772578800...",scalea-s-domenica-talao,NaN
6623144,IT,IC,1963,2026-03-04,8311775,2026-03-05 16:25:51.205+00,IC 1963,S11775,VILLA S.GIOVANNI MARE,2026-03-05 10:45:00+00,...,NaN,2026-03-05 11:05:00+00,2026-03-05 10:45:00+00,1200.0,NaN,NaN,NaN,"{""viaggiatrenoTripId"": ""S01700|1963|1772578800...",villa-s-giovanni-mare,NaN
6623146,IT,IC,1963,2026-03-04,8300180,2026-03-05 16:25:51.205+00,IC 1963,S06900,FIRENZE CAMPO MARTE,2026-03-04 23:40:30+00,...,NaN,2026-03-04 23:50:00+00,2026-03-04 23:12:00+00,2280.0,4,4,NaN,"{""viaggiatrenoTripId"": ""S01700|1963|1772578800...",firenze-campo-marte,NaN


In [20]:
len(chuuchuu_merge[chuuchuu_merge["country"].isna()])+country_counts["parent_count"].sum()

np.int64(6623163)

In [21]:
chuuchuu_merge_missing_countries = chuuchuu_merge[chuuchuu_merge["country"].isna()]
chuuchuu_merge_missing_countries.head()

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,stopName_slug,country
46,NS,Intercity,3142,2026-02-26,8400319,2026-02-25 13:23:47+00,Intercity 3142,3670859,'s-Hertogenbosch,2026-02-27 11:54:00+00,...,f,2026-02-27 11:55:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN
81,NS,Sprinter,8637,2026-02-26,8400053,2026-02-25 13:31:47+00,Sprinter 8637,2992146,Alphen a/d Rijn,NaN,...,f,2026-02-27 12:10:00+00,2026-02-26 12:10:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:R-NET_NS"", ""unplanned"": fals...",alphen-a/d-rijn,NaN
127,NS,Intercity,3171,2026-02-26,8400319,2026-02-25 13:31:47+00,Intercity 3171,3670859,'s-Hertogenbosch,2026-02-27 19:06:00+00,...,f,2026-02-27 19:08:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN
136,NS,Stoptrein,2583,2026-02-26,8800062,2026-02-25 13:31:47+00,Stoptrein 2583,2993451,Heide (B),2026-02-27 11:45:00+00,...,f,2026-02-27 11:46:00+00,2026-02-26 11:46:00+00,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NMBS"", ""unplanned"": false, ""...",heide-b,NaN
145,NS,Intercity,3143,2026-02-26,8400319,2026-02-25 13:31:47+00,Intercity 3143,3670859,'s-Hertogenbosch,2026-02-27 12:06:00+00,...,f,2026-02-27 12:08:00+00,NaN,0.0,NaN,NaN,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": true, ""ful...",'s-hertogenbosch,NaN


In [22]:
stations_unique[stations_unique["db_id"]!="<NA>"]

,id,name,slug,uic,uic8_sncf,latitude,longitude,parent_station_id,hub_id,country,...,info:ja,info:ko,info:pl,info:pt,info:ru,info:sv,info:tr,info:zh,normalised_code,iata_airport_code
0,1,Château-Arnoux – St-Auban,chateau-arnoux-st-auban,NaN,NaN,44.081790,6.001625,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv1,NaN
2,3,Château-Arnoux Mairie,chateau-arnoux-mairie,8775122.0,87751222.0,44.063863,6.011248,1.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv3,NaN
3,4,Digne-les-Bains,digne-les-bains,NaN,NaN,44.350000,6.350000,NaN,NaN,FR,...,ディーニュ＝レ＝バン,디뉴레뱅,NaN,NaN,Динь-ле-Бен,NaN,NaN,迪涅萊班,urn:trainline:public:nloc:csv4,NaN
5,7,La Crau,la-crau,8775561.0,87755611.0,43.144935,6.068766,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,Ла-Кро,NaN,NaN,拉克罗,urn:trainline:public:nloc:csv7,NaN
6,8,Aire-sur-l’Adour,aire-sur-ladour,8767104.0,87671040.0,43.703854,-0.258269,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv8,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70839,75060,Le Mans – Hôpital-Université,le-mans-hopital-universite,8774387.0,87743872.0,48.014871,0.174451,4661.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv75060,NaN
70840,75061,Rosa Parks,rosa-parks,8765479.0,87654798.0,48.896020,2.373970,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv75061,NaN
70841,75062,La Défense – Grande Arche RER E,la-defense-grande-arche-rer-e,8773143.0,87731430.0,48.892490,2.237790,9817.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv75062,NaN
70842,75063,Paris Gare de Lyon – Noctilien,paris-gare-de-lyon-noctilien,8758820.0,87588202.0,48.845780,2.373720,4924.0,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urn:trainline:public:nloc:csv75063,NaN


In [23]:
len(stations_unique)

69964

In [24]:
len(stations_unique["db_id"].unique())

29475

### Continue adding countries do the Chuuchuu data

In [25]:
if stations_unique["db_id"].dtype == "float64":
    stations_unique["db_id"] = stations_unique["db_id"].astype("Int64")
    stations_unique["db_id"] = stations_unique["db_id"].astype("str")

chuuchuu_merge["deutscheBahnStopId"] = chuuchuu_merge["deutscheBahnStopId"].astype("str")

stations_unique_db = stations_unique[stations_unique["db_id"]!="<NA>"]

# Create a mapping: db_id -> country
db_to_country = stations_unique_db.set_index("db_id")["country"]

# Replace country where there’s a match
chuuchuu_merge["country"] = chuuchuu_merge["deutscheBahnStopId"].map(db_to_country).fillna(
    chuuchuu_merge["country"]
)

C:\Users\TE\AppData\Local\Temp\ipykernel_2872\3515078428.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stations_unique["db_id"] = stations_unique["db_id"].astype("Int64")
C:\Users\TE\AppData\Local\Temp\ipykernel_2872\3515078428.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stations_unique["db_id"] = stations_unique["db_id"].astype("str")


In [26]:
chuuchuu_merge.head()

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,stopName_slug,country
0,NS,Sprinter,8148,2026-02-26,8400058,2026-02-25 04:51:47+00,Sprinter 8148,2992170,Amsterdam Centraal,2026-02-26 14:25:00+00,...,f,NaN,2026-02-26 14:25:00+00,NaN,10a,10a,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",amsterdam-centraal,NL
1,NS,Sprinter,8148,2026-02-26,8400059,2026-02-25 04:51:47+00,Sprinter 8148,2992236,Amsterdam Sloterdijk,2026-02-26 14:19:00+00,...,f,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00,0.0,12,12,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",amsterdam-sloterdijk,NL
2,NS,Sprinter,8148,2026-02-26,8400561,2026-02-25 04:51:47+00,Sprinter 8148,3065389,Schiphol Airport,2026-02-26 14:07:00+00,...,f,2026-02-26 14:09:00+00,2026-02-26 14:09:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",schiphol-airport,NL
3,NS,Sprinter,8148,2026-02-26,8400079,2026-02-25 04:51:47+00,Sprinter 8148,2860888,Amsterdam Lelylaan,2026-02-26 14:15:00+00,...,f,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00,0.0,2,2,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",amsterdam-lelylaan,NL
4,NS,Sprinter,8148,2026-02-26,8400332,2026-02-25 04:51:47+00,Sprinter 8148,2993022,Hoofddorp,NaN,...,f,2026-02-26 14:02:00+00,2026-02-26 14:02:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",hoofddorp,NL


In [27]:
chuuchuu_merge["country"].unique()

array(['NL', 'IT', 'BE', 'DK', 'DE', nan, 'FR', 'LU', 'CH', 'HU', 'AT',
       'SE', 'PL', 'HR', 'CZ', 'RS', 'SI', 'GB', 'UA', 'ES', 'RO', 'SK',
       'NO', 'LI'], dtype=object)

In [29]:
new_country_counts = (
    chuuchuu_merge.groupby("country")
    .agg(parent_count=("agency", lambda x: x.notna().sum()))
    .reset_index()
)
display(new_country_counts)
print(new_country_counts["parent_count"].sum())

,country,parent_count
0,AT,42832
1,BE,300963
2,CH,442880
3,CZ,3057
4,DE,3477556
5,DK,321582
6,ES,285
7,FR,456906
8,GB,569
9,HR,567


6259649


In [30]:
#Checking nb NA + nb country match vs total length dataset
print(len(chuuchuu_merge[chuuchuu_merge["country"].isna()])+(new_country_counts["parent_count"].sum()))
print(len(chuuchuu_merge))

#checking the % of NA:
a = round((len(chuuchuu_merge[chuuchuu_merge["country"].isna()])/len(chuuchuu))*100,2)
print(f"{a}%")
# 5.63%

6623163
6623163
5.49%


In [31]:
chuuchuu_merge[chuuchuu_merge["country"].isna()]

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,stopName_slug,country
2180,DK,IC,70182,2026-02-26,8600506,2026-02-25 23:19:17.155+00,IC 70182,8600506,Aarup St.,2026-02-25 23:50:00+00,...,f,2026-02-25 23:50:00+00,2026-02-25 23:50:00+00,0.0,NaN,1,f,"{""operator"": ""DSB"", ""data_source"": ""dsb"", ""pro...",aarup-st,NaN
2192,DK,Re,74895,2026-02-26,8600798,2026-02-25 23:19:17.155+00,Re 74895,8600798,Høje Taastrup St.,NaN,...,f,2026-02-25 23:33:00+00,2026-02-25 23:33:00+00,0.0,NaN,4,f,"{""operator"": ""DSB"", ""data_source"": ""dsb"", ""pro...",hje-taastrup-st,NaN
2209,DK,Togbus,99388,2026-02-26,8650044,2026-02-25 23:19:17.155+00,Togbus,8650044,Langå St. (togbus),NaN,...,f,NaN,2026-02-25 23:32:00+00,NaN,NaN,NaN,f,"{""operator"": ""DSB"", ""data_source"": ""dsb"", ""pro...",langa-st-togbus,NaN
2834,DK,IC,71472,2026-02-26,8600858,2026-02-26 00:02:39.601+00,IC 71472,8600858,København Lufthavn,2026-02-26 00:01:00+00,...,f,NaN,NaN,NaN,NaN,NaN,f,"{""operator"": ""DSB"", ""data_source"": ""dsb"", ""pro...",kbenhavn-lufthavn,NaN
2840,DK,Togbus,99388,2026-02-26,8650047,2026-02-26 00:02:39.601+00,Togbus,8650047,Stop 8650047,2026-02-26 00:02:00+00,...,f,2026-02-26 00:02:00+00,2026-02-26 00:02:00+00,0.0,NaN,NaN,f,"{""operator"": ""DSB"", ""data_source"": ""dsb"", ""pro...",stop-8650047,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6622994,HU,IC,78,2026-03-04,5310988,2026-03-05 11:49:20.537+00,IC 78,1:005310988_0,Strehaia Halta,2026-03-05 00:08:00+00,...,NaN,2026-03-05 00:09:00+00,2026-03-05 00:09:00+00,0.0,NaN,NaN,NaN,"{""tripId"": ""VHJpcDoxOjMwMDE4OTc1"", ""routeName""...",strehaia-halta,NaN
6622996,HU,IC,78,2026-03-04,5311140,2026-03-05 11:49:20.537+00,IC 78,1:005311140_0,Drobeta Tr.Severin,2026-03-05 01:26:00+00,...,NaN,2026-03-05 01:27:00+00,2026-03-05 01:27:00+00,0.0,NaN,NaN,NaN,"{""tripId"": ""VHJpcDoxOjMwMDE4OTc1"", ""routeName""...",drobeta-tr-severin,NaN
6622999,HU,IC,78,2026-03-04,5517822,2026-03-05 11:49:20.537+00,IC 78,1:005517822_B,Kétpó,2026-03-05 10:09:00+00,...,NaN,2026-03-05 10:10:00+00,2026-03-05 10:08:00+00,120.0,B,B,NaN,"{""tripId"": ""VHJpcDoxOjMwMDE4OTc1"", ""routeName""...",ketpo,NaN
6623023,IT,IC,1959,2026-03-04,8311775,2026-03-05 12:23:20.399+00,IC 1959,S11775,VILLA S.GIOVANNI MARE,2026-03-05 07:30:00+00,...,NaN,2026-03-05 07:50:00+00,2026-03-05 07:15:00+00,2100.0,NaN,NaN,NaN,"{""viaggiatrenoTripId"": ""S08409|1959|1772578800...",villa-s-giovanni-mare,NaN


In [32]:
chuuchuu_merge["agency"][chuuchuu_merge["country"].isna()].unique()

array(['DK', 'HU', 'NMBS', 'IT', 'GTFSDE', 'PL', 'NS', 'FR', 'DB', 'EST',
       'SBB'], dtype=object)

In [33]:
agency_counts_missing_countries = (
    chuuchuu_merge[chuuchuu_merge["country"].isna()].groupby("agency")
    .agg(parent_count=("agency", lambda x: x.notna().sum()))
    .reset_index()
)
display(agency_counts_missing_countries)
print(agency_counts_missing_countries["parent_count"].sum())

,agency,parent_count
0,DB,4389
1,DK,66106
2,EST,170
3,FR,15493
4,GTFSDE,14188
5,HU,227852
6,IT,11257
7,NMBS,4365
8,NS,152
9,PL,5668


363514


In [ ]:
#According to Chuuchuu, we know for sure that GTFSDE = Germany --> I disagree

In [34]:
HU = chuuchuu_merge[(chuuchuu_merge["country"].isna()) & (chuuchuu_merge["agency"]=="HU")]

In [35]:
print(len(HU["stopName"].unique()))

print(list(HU["stopName"].unique()))

1034
['Rákos', 'Kőbánya felső', 'Akadémiaújtelep', 'Máriabesnyő', 'Isaszeg', 'Rákosliget', 'Gödöllő', 'Rákoscsaba-Újtelep', 'Rákoscsaba', 'Pécel', 'Galgahévíz', 'Hévízgyörk', 'Aszód', 'Velencefürdő', 'Gárdony', 'Agárd', 'Rákospalota-Újpest', 'Alagimajor', 'Fót', 'Fótújfalu', 'Csomád', 'Veresegyház', 'Kápolnásnyék', 'Ivacs', 'Váchartyán', 'Fótfürdő', 'Rákosrendező', 'Rudnaykert', 'Máriaudvar', 'Vicziántelep', 'Rákospalota-Kertváros', 'Vác-Alsóváros', 'Csörög', 'Istvántelek', 'Őrbottyán', 'Barosstelep', 'Erdőkertes', 'Budatétény', 'Érd felső', 'Érdliget', 'Dinnyés', 'Baracska', 'Martonvásár', 'Tárnok', 'Budafok', 'Velence', 'Pettend', 'Solymár', 'Szélhegy', 'Vörösvárbánya', 'Pilisvörösvár', 'Aranyvölgy', 'Újpest', 'Esztergom-Kertváros', 'Piliscsaba', 'Leányvár', 'Szabadságliget', 'Piliscsév', 'Aquincum', 'Magdolnavölgy', 'Pilisjászfalu', 'Kispest', 'Kőbánya alsó', 'Felsőlajos', 'Táborfalva', 'Örkény', 'Hernád', 'Ócsa', 'Gyál', 'Gyál felső', 'Felsőpakony', 'Lajosmizse', 'Pestszentimre', '

### Continue adding countries to the Chuuchuu data using stations info I get on OSM

In [36]:
# Create a mapping: db_id -> country
osm_to_country = osm_stations.set_index("name_slug")["country"]

# Replace country where there’s a match
chuuchuu_merge["country"] = chuuchuu_merge["stopName_slug"].map(db_to_country).fillna(
    chuuchuu_merge["country"]
)

In [37]:
#Checking nb NA + nb country match vs total length dataset
print(len(chuuchuu_merge[chuuchuu_merge["country"].isna()])+(new_country_counts["parent_count"].sum()))
print(len(chuuchuu_merge))

#checking the % of NA:
a = round((len(chuuchuu_merge[chuuchuu_merge["country"].isna()])/len(chuuchuu))*100,2)
print(f"{a}%")

6623163
6623163
5.49%


Definitely not changing: waste of time

Fuzzy match? Using AI to match, similarly to topic categorization on the CEF report? Let's check this next week

In [53]:
#Identify the rows with ocuntry still NA
to_fix = chuuchuu_merge[chuuchuu_merge['country'].isna()].copy()
fixed_rows = chuuchuu_merge[chuuchuu_merge['country'].notna()].copy()
unique_unmatched_stations = to_fix['stopName_slug'].unique()

In [56]:
# trying a fuzzy match
from rapidfuzz import process, distance

In [57]:
# 1. Separate the data
to_fix = chuuchuu_merge[chuuchuu_merge['country'].isna()].copy()
fixed_rows = chuuchuu_merge[chuuchuu_merge['country'].notna()].copy()

# 2. Prepare the lists (cleaning any unexpected NaNs or non-strings)
unique_unmatched_stations = [str(x) for x in to_fix['stopName_slug'].unique() if pd.notna(x)]
choices = [str(x) for x in stations_unique["slug"].tolist() if pd.notna(x)]

if unique_unmatched_stations and choices:
    # 3. Perform the bulk match (Fastest way)
    # normalized_similarity returns a 0.0 to 1.0 float
    matrix = process.cdist(
        unique_unmatched_stations, 
        choices, 
        scorer=distance.Levenshtein.normalized_similarity, 
        workers=-1
    )

    # 4. Extract best match and its score
    best_match_indices = matrix.argmax(axis=1)
    
    mapping = {}
    for i, typo in enumerate(unique_unmatched_stations):
        idx = best_match_indices[i]
        score = matrix[i, idx]
        
        # Only map if similarity is > 80% (0.8)
        if score > 0.8:
            mapping[typo] = choices[idx]
        else:
            mapping[typo] = None

    # 5. Map the corrections back to the 'to_fix' dataframe
    to_fix['slug_corrected'] = to_fix['stopName_slug'].map(mapping)

    # 6. Re-merge the fixed rows to get the country
    # We drop the old empty 'country' column first
    to_fix_merged = to_fix.drop(columns=['country']).merge(
        stations_unique[["slug", "country"]],
        left_on="slug_corrected",
        right_on="slug",
        how="left"
    ).drop(columns=["slug", "slug_corrected"])

    # 7. Combine back with the rows that were originally fine
    final_df = pd.concat([fixed_rows, to_fix_merged], ignore_index=True)
else:
    final_df = chuuchuu_merge.copy()

# Final cleanup: check how many rows are still NA
print(f"Remaining NAs: {final_df['country'].isna().sum()}")

Remaining NAs: 338174


In [58]:
#checking the % of NA:
a = round((len(final_df[final_df["country"].isna()])/len(final_df))*100,2)
print(f"{a}%")

5.11%


In [59]:
len(final_df)

6623163

Email from chuuchuu: For 2025 Wrapped we didn't include data for Hungary, Poland or Denmark yet, while for other countries we matched the country based on the first two numbers of the deutscheBahnStopId.

In [61]:
agencies_to_drop = ["DK", "PL", "HU"]

final_df = final_df[~final_df["agency"].isin(agencies_to_drop)]

In [62]:
#checking the % of NA:
a = round((len(final_df[final_df["country"].isna()])/len(final_df))*100,2)
print(f"{a}%")

0.88%
